# Polychromatic LGS with 589 + 569 nm pump

In [ ]:
# import os
# os.environ['OMP_NUM_THREADS'] = '1'
# os.environ['OPENBLAS_NUM_THREADS'] = '1'
# os.environ['MKL_NUM_THREADS'] = '1'

In [ ]:
from pylgs.all import *

In [ ]:
import logging
logger = logging.getLogger().setLevel(logging.INFO)

Define a LGS system pumped by 330 nm light:

In [ ]:
lgs = LGSSystem(
    'Na589+569', 
    {
        # 'IntensitySI1': 50.,
        'EllipticityDegrees1': 45.,
        'PolarizationAngleDegrees1': 0,
        'DetuningHz1': 1.0832e9,
        'LaserWidthHz1': 10.0e6,
        # 'IntensitySI2': 50000.,
        'EllipticityDegrees2': 45.,
        'PolarizationAngleDegrees2': 0,
        'DetuningHz2': 1e8,
        'LaserWidthHz2': 10.0e6,
        'BFieldG': 0.5,
        'MagneticZenithDegrees': 45.,
        'MagneticAzimuthDegrees': 45.,
        'SDampingCollisionRatePerS': 4081.63,
        'BeamTransitRatePerS': 131.944,
        'VccRatePerS': 28571.,
        'TemperatureK': 185.,
        'RecoilParameter': 1.
    }
)

## Steady-state model

### Adaptively refine velocity groups

Build a steady-state model with adaptively refined velocity groups:

In [ ]:
model, sol = lgs.adaptive_stationary_model({'IntensitySI1': 50., 'IntensitySI2': 1e3}, rtol=.01)

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

Plot the total return flux for each transition as a bar chart:

In [ ]:
elems = sol.array.sum('Atomic velocity')
pops = elems.where(population_element(elems['Density matrix']), drop=True)
plotly_dataarray(pops.sortby(pops)[-15:], log_x=True)

In [ ]:
model.total_flux(sol).visualize(log_x=True)

In [ ]:
sol = model.solve([{}, {'IntensitySI2': 10**np.linspace(1, 4, 20)}])

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [ ]:
model.total_flux(sol).visualize(log_x=True, log_y=True)

In [ ]:
model.flux_distribution(sol).visualize(xaxis_range=(-.5, .5))

In [ ]:
results = []
for detuning in np.linspace(-.2e9, .2e9, 5):
    model, sol = lgs.adaptive_stationary_model({'DetuningHz2': detuning}, rtol=.005)
    results.append([detuning, model.total_flux(sol).array.data[0]])
results

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

[[np.float64(-200000000.0), array([212.20296145,   6.0276221 ])],
 [np.float64(-100000000.0), array([164.10072659,   9.08874614])],
 [np.float64(0.0), array([134.20862658,  11.65697361])],
 [np.float64(100000000.0), array([139.99519875,  10.14101316])],
 [np.float64(200000000.0), array([160.38768051,   9.20976745])]]

In [ ]:
model.total_flux(sol).visualize()

Plot the flux as a function of velocity:

In [ ]:
model.flux_distribution(sol).visualize(xaxis_range=(-1, 2))

In [ ]:
sol = model.solve([{}, {'DetuningHz2': np.linspace(-.2e9, .2e9, 50)}])

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize()

In [ ]:
model.total_flux(sol).visualize(log_y=True)

In [ ]:
model.total_flux(sol).visualize(log_y=True)

In [ ]:
model.total_flux(sol).visualize(log_y=True)

Create an animation of the flux distribution as the repump intensity is varied:

In [ ]:
model.flux_distribution(sol).visualize(xaxis_range=(-.5, .5))

Animate the level population distribution as a function of repump intensity:

In [ ]:
model.level_population_distribution(sol).visualize()

## Transient dynamics model

Build a model for the transient dynamics:

In [ ]:
model = lgs.instationary_model(
    vg=model.data['velocity_groups'], 
    T=1e-6, 
    num_values=101
)

Set tolerances for the BDF solver:

In [ ]:
pymor.basic.set_defaults({
    'pylgs.pymor.timestepping.cvode_solver_options.cvode_bdf_atol': 1e-6,
    'pylgs.pymor.timestepping.cvode_solver_options.cvode_bdf_rtol': 1e-5,
})

Solve for the density matrix as a function of time:

In [ ]:
sol = model.solve()

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

Plot the return flux for each transition:

In [ ]:
model.total_flux(sol).visualize()

Animate the level population distribution as a function of time:

In [ ]:
model.level_population_distribution(sol).visualize(xaxis_range=(-.25, 2), yaxis_range=(0, .045))